<a href="https://colab.research.google.com/github/anabiarochar/Mestrado_AlgoritmosProgramacao/blob/main/Projeto2/Projeto2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import pandas as pd
!pip install gdown

In [26]:
# Google Drive - ID do arquivo
file_id = '1fsbd9Zm7h9k4oxa0baM0EJ7ChAZhiSir'

# Arquivo Output
output_filename = 'dataset.csv'

# Download do arquivo usando gdown
!gdown --id {file_id} -O {output_filename}

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1fsbd9Zm7h9k4oxa0baM0EJ7ChAZhiSir
To: /content/dataset.csv
100% 6.71M/6.71M [00:00<00:00, 28.0MB/s]


In [27]:
df = pd.read_csv(output_filename)
display(df.head())

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,821198084644106078,Bright and Peaceful Leblon Loft,84350716,Katrina,NaN,Leblon,-22.982818,-43.222457,Entire home/apt,580.0,2,86,2025-09-19,2.72,1,82,40,NaN
1,821198370698658112,Copacabana 100% reformado.,4347269,Patrick,NaN,Copacabana,-22.984090,-43.191770,Entire home/apt,1900.0,5,0,NaN,NaN,2,364,0,NaN
2,821200521820144734,hambiente familia,499903412,Vanessa,NaN,Pavuna,-22.814911,-43.379011,Entire home/apt,700.0,1,0,NaN,NaN,2,365,0,NaN
3,821213014263313420,Amazing en suite bedroom Leblon,449763717,Alvaro,NaN,Leblon,-22.981910,-43.225990,Private room,NaN,2,3,2024-05-02,0.10,2,0,0,NaN
4,821223043903573522,Incrível apartamento frente mar,25961210,Katia,NaN,Barra da Tijuca,-23.010000,-43.344820,Entire home/apt,500.0,2,11,2025-08-26,0.60,1,234,10,NaN


### Preparação dos dados para criar as árvores

Coluna 'price' será a key para ambas a Árvore Binária de Busca e para a AVL. Nessa etapa removeremos as linhas com preço `NaN`.

In [28]:
df_cleaned = df.dropna(subset=['price']).copy()

# Converter 'price' para float
df_cleaned['price'] = df_cleaned['price'].astype(float)

print(f"DataFrame Original: {df.shape}")
print(f"DataFrame Tratado (sem NaN prices): {df_cleaned.shape}")

display(df_cleaned.head())

DataFrame Original: (43068, 18)
DataFrame Tratado (sem NaN prices): (38670, 18)


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,821198084644106078,Bright and Peaceful Leblon Loft,84350716,Katrina,NaN,Leblon,-22.982818,-43.222457,Entire home/apt,580.0,2,86,2025-09-19,2.72,1,82,40,NaN
1,821198370698658112,Copacabana 100% reformado.,4347269,Patrick,NaN,Copacabana,-22.984090,-43.191770,Entire home/apt,1900.0,5,0,NaN,NaN,2,364,0,NaN
2,821200521820144734,hambiente familia,499903412,Vanessa,NaN,Pavuna,-22.814911,-43.379011,Entire home/apt,700.0,1,0,NaN,NaN,2,365,0,NaN
4,821223043903573522,Incrível apartamento frente mar,25961210,Katia,NaN,Barra da Tijuca,-23.010000,-43.344820,Entire home/apt,500.0,2,11,2025-08-26,0.60,1,234,10,NaN
5,821227099948297881,Apt N S de Copacabana reformado!,392769753,Amanda,NaN,Copacabana,-22.970696,-43.186048,Entire home/apt,461.0,2,89,2025-09-21,2.81,7,113,41,NaN


### Implementando a Árvore Binária de Busca (BST)

In [29]:
class BSTNode:
    def __init__(self, key, data):
        self.key = key
        self.data = [data] # Armazenar dados em uma lista para lidar com chaves duplicadas
        self.left = None
        self.right = None

class BinarySearchTree:
    def __init__(self):
        self.root = None

    def insert(self, key, data):
        new_node_data = {
            'name': data.get('name'),
            'host_name': data.get('host_name'),
            'neighbourhood': data.get('neighbourhood'),
            'room_type': data.get('room_type'),
            'number_of_reviews': data.get('number_of_reviews'),
            'availability_365': data.get('availability_365')
        }
        if self.root is None:
            self.root = BSTNode(key, new_node_data)
        else:
            self._insert_recursive(self.root, key, new_node_data)

    def _insert_recursive(self, node, key, data):
        if key < node.key:
            if node.left is None:
                node.left = BSTNode(key, data)
            else:
                self._insert_recursive(node.left, key, data)
        elif key > node.key:
            if node.right is None:
                node.right = BSTNode(key, data)
            else:
                self._insert_recursive(node.right, key, data)
        else: # Lidar com chaves duplicadas anexando dados ao nó existente
            node.data.append(data)

    def inorder_traversal(self):
        items = []
        self._inorder_recursive(self.root, items)
        return items

    def _inorder_recursive(self, node, items):
        if node:
            self._inorder_recursive(node.left, items)
            for data_item in node.data:
                items.append((node.key, data_item))
            self._inorder_recursive(node.right, items)

    def search(self, key):
        return self._search_recursive(self.root, key)

    def _search_recursive(self, node, key):
        if node is None or node.key == key:
            return node
        if key < node.key:
            return self._search_recursive(node.left, key)
        return self._search_recursive(node.right, key)

    def get_height(self):
        return self._get_height_recursive(self.root)

    def _get_height_recursive(self, node):
        if not node:
            return 0
        return 1 + max(self._get_height_recursive(node.left), self._get_height_recursive(node.right))

# Criar uma instância da BST
bst = BinarySearchTree()

# Popular a BST com dados do DataFrame limpo
for index, row in df_cleaned.iterrows():
    bst.insert(row['price'], row.to_dict())

print("BST criada e populada. Exemplo de percurso em ordem (primeiros 5 itens):")
# Exibir os 5 primeiros elementos do percurso em ordem para brevidade
for i, item in enumerate(bst.inorder_traversal()):
    if i >= 5:
        break
    print(item)

BST criada e populada. Exemplo de percurso em ordem (primeiros 5 itens):
(30.0, {'name': 'Quarto próximo ao Estádio Olímpico', 'host_name': 'Rafael', 'neighbourhood': 'Cachambi', 'room_type': 'Private room', 'number_of_reviews': 1, 'availability_365': 358})
(39.0, {'name': 'Quarto da vó Vina na Vila da Penha', 'host_name': 'Etelvina', 'neighbourhood': 'Vicente de Carvalho', 'room_type': 'Private room', 'number_of_reviews': 0, 'availability_365': 365})
(40.0, {'name': 'Quarto 303 vip', 'host_name': 'Marcelo', 'neighbourhood': 'Bonsucesso', 'room_type': 'Private room', 'number_of_reviews': 5, 'availability_365': 365})
(41.0, {'name': 'Pague o valor d/1 Aluguel conquiste sua Cs Propria', 'host_name': 'Diego Vitor', 'neighbourhood': 'Campo Grande', 'room_type': 'Entire home/apt', 'number_of_reviews': 0, 'availability_365': 365})
(42.0, {'name': 'Simples e amigável', 'host_name': 'Rogério', 'neighbourhood': 'Praça Seca', 'room_type': 'Private room', 'number_of_reviews': 1, 'availability_365

### Implementando uma Árvore AVL

In [30]:
class AVLNode:
    def __init__(self, key, data):
        self.key = key
        self.data = [data] # Armazenar dados em uma lista para lidar com chaves duplicadas
        self.left = None
        self.right = None
        self.height = 1 # Altura do nó (1 para um nó folha inicialmente)

class AVLTree:
    def __init__(self):
        self.root = None

    def _height(self, node):
        if not node:
            return 0
        return node.height

    def _balance_factor(self, node):
        if not node:
            return 0
        return self._height(node.left) - self._height(node.right)

    def _update_height(self, node):
        if node:
            node.height = 1 + max(self._height(node.left), self._height(node.right))

    def _rotate_right(self, y):
        x = y.left
        T2 = x.right

        # Realizar rotação
        x.right = y
        y.left = T2

        # Atualizar alturas
        self._update_height(y)
        self._update_height(x)

        return x

    def _rotate_left(self, x):
        y = x.right
        T2 = y.left

        # Realizar rotação
        y.left = x
        x.right = T2

        # Atualizar alturas
        self._update_height(x)
        self._update_height(y)

        return y

    def insert(self, key, data):
        new_node_data = {
            'name': data.get('name'),
            'host_name': data.get('host_name'),
            'neighbourhood': data.get('neighbourhood'),
            'room_type': data.get('room_type'),
            'number_of_reviews': data.get('number_of_reviews'),
            'availability_365': data.get('availability_365')
        }
        self.root = self._insert_recursive(self.root, key, new_node_data)

    def _insert_recursive(self, node, key, data):
        # 1. Realizar inserção padrão da BST
        if not node:
            return AVLNode(key, data)

        if key < node.key:
            node.left = self._insert_recursive(node.left, key, data)
        elif key > node.key:
            node.right = self._insert_recursive(node.right, key, data)
        else: # Chaves duplicadas
            node.data.append(data)
            return node

        # 2. Atualizar altura do nó atual
        self._update_height(node)

        # 3. Obter o fator de balanceamento
        balance = self._balance_factor(node)

        # 4. Realizar rotações se desbalanceado
        # Caso Esquerda Esquerda
        if balance > 1 and key < node.left.key:
            return self._rotate_right(node)

        # Caso Direita Direita
        if balance < -1 and key > node.right.key:
            return self._rotate_left(node)

        # Caso Esquerda Direita
        if balance > 1 and key > node.left.key:
            node.left = self._rotate_left(node.left)
            return self._rotate_right(node)

        # Caso Direita Esquerda
        if balance < -1 and key < node.right.key:
            node.right = self._rotate_right(node.right)
            return self._rotate_left(node)

        return node

    def inorder_traversal(self):
        items = []
        self._inorder_recursive(self.root, items)
        return items

    def _inorder_recursive(self, node, items):
        if node:
            self._inorder_recursive(node.left, items)
            for data_item in node.data:
                items.append((node.key, data_item))
            self._inorder_recursive(node.right, items)

    def get_height(self):
        return self._height(self.root)

# Criar uma instância da árvore AVL
avl_tree = AVLTree()

# Popular a árvore AVL com dados do DataFrame limpo
for index, row in df_cleaned.iterrows():
    avl_tree.insert(row['price'], row.to_dict())

print("\nÁrvore AVL criada e populada. Exemplo de percurso em ordem (primeiros 5 itens):")
# Exibir os 5 primeiros elementos do percurso em ordem para brevidade
for i, item in enumerate(avl_tree.inorder_traversal()):
    if i >= 5:
        break
    print(item)


Árvore AVL criada e populada. Exemplo de percurso em ordem (primeiros 5 itens):
(30.0, {'name': 'Quarto próximo ao Estádio Olímpico', 'host_name': 'Rafael', 'neighbourhood': 'Cachambi', 'room_type': 'Private room', 'number_of_reviews': 1, 'availability_365': 358})
(39.0, {'name': 'Quarto da vó Vina na Vila da Penha', 'host_name': 'Etelvina', 'neighbourhood': 'Vicente de Carvalho', 'room_type': 'Private room', 'number_of_reviews': 0, 'availability_365': 365})
(40.0, {'name': 'Quarto 303 vip', 'host_name': 'Marcelo', 'neighbourhood': 'Bonsucesso', 'room_type': 'Private room', 'number_of_reviews': 5, 'availability_365': 365})
(41.0, {'name': 'Pague o valor d/1 Aluguel conquiste sua Cs Propria', 'host_name': 'Diego Vitor', 'neighbourhood': 'Campo Grande', 'room_type': 'Entire home/apt', 'number_of_reviews': 0, 'availability_365': 365})
(42.0, {'name': 'Simples e amigável', 'host_name': 'Rogério', 'neighbourhood': 'Praça Seca', 'room_type': 'Private room', 'number_of_reviews': 1, 'availabi

In [31]:
print(f"Altura da BST: {bst.get_height()}")
print(f"Altura da AVL: {avl_tree.get_height()}")

Altura da BST: 25
Altura da AVL: 14


In [32]:
num_unique_bst_nodes = len(bst.inorder_traversal())
num_unique_avl_nodes = len(avl_tree.inorder_traversal())

print(f"Número de nós únicos na BST (chaves de preço distintas): {num_unique_bst_nodes}")
print(f"Número de nós únicos na AVL (chaves de preço distintas): {num_unique_avl_nodes}")

Número de nós únicos na BST (chaves de preço distintas): 38670
Número de nós únicos na AVL (chaves de preço distintas): 38670
